# 05 — Evaluation

Run all 4 configs (A/B/C/D) on the test set, compute BLEU / ROUGE-L / BERTScore / Recall@5 / MRR@10, and build the 50-question human-eval form.

Run order:
1. Phase 2 → `scripts/build_index.py` (FAISS index)
2. Phase 3 → fine-tune adapter + push to `Tamir39/qwen2_5-7b-vietnam-tax-lora`
3. **This notebook** → `scripts/run_eval.py` then `scripts/build_human_eval.py`

In [ ]:
import sys, subprocess
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import INDEX_DIR, RESULTS_DIR
print('index ready:', (INDEX_DIR / 'kb.faiss').exists())
print('results dir:', RESULTS_DIR)

In [ ]:
# Run the 4-config matrix on the test set. ~6-12 min per config on P100.
# Drop --skip-bertscore once the run is stable; BERTScore needs xlm-roberta-large.
subprocess.check_call([
    sys.executable, str(ROOT / 'scripts' / 'run_eval.py'),
])

In [ ]:
import json
import pandas as pd

summary = json.loads((RESULTS_DIR / 'summary.json').read_text(encoding='utf-8'))
df = pd.DataFrame(summary).T
df.index.name = 'config'
df

In [ ]:
# Build the blinded 50-question human-eval form.
subprocess.check_call([
    sys.executable, str(ROOT / 'scripts' / 'build_human_eval.py'), '--n', '50',
])

## After rating

1. Open `experiments/results/human_eval/form.csv` in any editor / spreadsheet.
2. Fill `rating_1..rating_4` with integers 1-5 (1 = wrong / hallucinated, 5 = perfect grounded answer).
3. Optionally add notes.
4. Join with `key.csv` on `(row_id, slot)` to recover the underlying config and report mean rating per config (A/B/C/D).